# Cattle Re-ID — Runner de Kaggle (Fases 0→4)

Orquestador para correr el pipeline en **Kaggle con GPU**. La lógica vive en `src/` y `scripts/`; este notebook solo orquesta (ver `CLAUDE.md`).

**Antes de correr — configurar el notebook:**
1. *Settings → Accelerator → **GPU T4 ×2***. **No usar P100**: la imagen actual de Kaggle trae un PyTorch que dejó de soportar Pascal/sm_60 → `CUDA error: no kernel image is available`. La T4 es sm_75, sí soportada. (El paper usó P100; ver `DEVIATIONS.md`.)
2. *Settings → Internet → **ON*** (necesario para `git clone`).
3. *Add Input* → adjuntar **dos** datasets:
   - El de **imágenes** (contiene `BeefCattle_Muzzle_Individualized/`).
   - El de **pesos ImageNet** (`vgg16_bn-6c64b313.pth`, `resnet50-0676ba61.pth`, sin renombrar).

`config.py` autodetecta ambos en `/kaggle/input/` (búsqueda recursiva, sirve aunque el mount quede en `/kaggle/input/datasets/<user>/<slug>/...`). Todo lo que se escribe en `/kaggle/working/` persiste al hacer **Save Version**.

> **Para el sweep largo usar "Save & Run All" (commit)**: corre en background hasta ~12 h sin estar conectado. La sesión interactiva se corta por idle.

## 0. Verificar GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No hay GPU: Settings -> Accelerator -> GPU T4 x2 (no P100)'
# Sanity: la GPU tiene que ser compatible con el PyTorch de la imagen (T4 = sm_75 OK; P100 = sm_60 NO).
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU: {name} | capability sm_{cap[0]}{cap[1]}')
    if cap[0] < 7:
        print('AVISO: esta GPU (sm_<70) puede no ser compatible con el PyTorch de Kaggle. '
              'Si el forward tira "no kernel image", cambiá a GPU T4 x2.')

## 1. Traer el código (git clone / pull)

In [ ]:
import os

REPO_URL = 'https://github.com/benjaminvitale/tp-final-vision2-Pujol-Vitale.git'
REPO_DIR = '/kaggle/working/tp-final-vision2-Pujol-Vitale'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

%cd {REPO_DIR}
!git log --oneline -1

## 2. Verificar rutas (DATA_DIR + PRETRAINED_DIR)

`config.py` resuelve el dataset y los pesos desde `/kaggle/input/`. **No avanzar** si `DATA_DIR` no existe o `PRETRAINED_DIR` es `None` (sin los `.pth`, los modelos intentarán bajarlos por internet).

In [ ]:
import config, importlib
importlib.reload(config)

print('DATA_DIR       :', config.DATA_DIR, '| existe:', config.DATA_DIR.is_dir())
print('PRETRAINED_DIR :', config.PRETRAINED_DIR)

assert config.DATA_DIR.is_dir(), 'DATA_DIR no encontrado: adjuntá el dataset de imágenes (Add Input).'
if config.PRETRAINED_DIR is None:
    print('AVISO: no se encontraron los .pth de ImageNet. Se intentarán bajar por internet '
          '(Internet debe estar ON). Recomendado: adjuntar el dataset de pesos.')

## 3. Fase 0 — Inspección de datos (SIEMPRE primero)

Sanity check contra el paper: **268 clases, 4923 imágenes, 4–70 por clase**. No avanzar si no cuadra.

In [ ]:
!python scripts/00_inspect_data.py

## 4. Splits (ya commiteados — NO regenerar)

Los splits viven en `outputs/splits/` con rutas relativas a `DATA_DIR`, así que el mismo split sirve en Kaggle y local. Esta celda solo los **verifica**. Correr `01_make_splits.py` solo si querés regenerarlos a propósito (rompe la reproducibilidad con los commiteados).

In [ ]:
from src.utils import load_json
for name in ('train', 'val', 'test'):
    n = len(load_json(config.SPLITS_DIR / f'{name}.json'))
    print(f'{name:5}: {n} imágenes')
label_map = load_json(config.SPLITS_DIR / 'label_map.json')
print('clases en label_map:', len(label_map))

## 5. Smoke test en GPU (validar ANTES de quemar cuota)

Confirma que paths + GPU + pipeline resuelven end-to-end. Usa subset chico, 2 épocas y `pretrained=False` (no baja pesos). La accuracy va a dar ~0: es esperado, valida el pipeline, no la ciencia. ~1–2 min.

In [ ]:
!python scripts/02_train_vgg.py --smoke
!python scripts/03_train_resnet.py --smoke

## 6. Fase 3 — Replicación VGG16_BN (el deliverable principal)

3 variantes (`ce`, `ce_aug`, `wce`) × **3 semillas**, 50 épocas. **Este es el sweep largo** → correr con *Save & Run All* (background).

**Presupuesto de tiempo (medido en T4):** ~43 min/corrida → 9 corridas ≈ **6.5 h**. Sumando ResNet (celda 7) da ~9–9.5 h, entra en el límite de 12 h. El sweep VGG corre *antes* que ResNet, así que su summary + checkpoints quedan escritos primero.

> Las 3 semillas son `0,1,2` (ver `DEVIATIONS.md` D2: bajamos de 5 a 3 por el presupuesto de la T4; igual da media ± std). Para las 5 del plan: `--seeds 0 1 2 3 4`, pero **no entra en un solo commit** — habría que partirlo.

Validación de éxito: mejor variante ~96–98%+, y `ce_aug`/`wce` mejoran las clases con 4 imágenes vs `ce`. Resultado en `outputs/results/02_vgg_summary.json`.

In [ ]:
# 3 semillas para entrar en el presupuesto de 12 h en T4 (ver DEVIATIONS.md D2).
# Para las 5 del plan: --seeds 0 1 2 3 4 (no entra en un solo commit, hay que partirlo).
!python scripts/02_train_vgg.py --seeds 0 1 2

## 7. Fase 4 — Backbone propio ResNet-50

Dos modos: `freeze` (paper, solo FC) y `finetune` (full fine-tune). El mejor run por val accuracy se copia a `outputs/checkpoints/resnet50_backbone.pt` → backbone para domain adaptation.

> Backbone más fuerte para DA: agregar `--aug` y/o `--loss wce`. Más semillas: `--seeds 0 1 2`.

In [ ]:
!python scripts/03_train_resnet.py

## 8. Resultados y checkpoints (persisten al hacer Save Version)

In [ ]:
import json

print('=== outputs/results/ ===')
!ls -lh outputs/results/
print('\n=== outputs/checkpoints/ ===')
!ls -lh outputs/checkpoints/

for f in ('02_vgg_summary.json', '03_resnet_summary.json'):
    p = config.RESULTS_DIR / f
    if p.exists():
        print(f'\n=== {f} (summary) ===')
        data = json.loads(p.read_text())
        print(json.dumps(data.get('summary', data), indent=2, ensure_ascii=False)[:2000])